In [ ]:
# SHDI Manual Calculation - WORKING METHOD- takes 50 million years but works

import os
import glob
import time
import rasterio
import numpy as np
from scipy.ndimage import generic_filter
import gc

# PARAMETERS
INPUT_FOLDER = r"/home/gisuser/code/data/MB_v9_AtlanticForest_Landcover/orig_P_correct_nsf2_1.5_for_SHDI"
OUTPUT_FOLDER = r"/home/gisuser/code/data/SHDI_landcover"
RADIUS_M = 1000

def calculate_shdi(window):
    """Calculate Shannon Diversity Index for a window"""
    window = window[window != 0]
    if len(window) == 0:
        return 0
    unique, counts = np.unique(window, return_counts=True)
    proportions = counts / counts.sum()
    return -np.sum(proportions * np.log(proportions))

def process_file(raster_path, output_dir, radius_m=1000):
    filename = os.path.basename(raster_path)
    output_path = os.path.join(output_dir, filename.replace('.tif', '_shdi.tif'))
    
    if os.path.exists(output_path):
        print(f"Skipping {filename} - already exists")
        return output_path
    
    print(f"\n{'='*60}")
    print(f"Processing: {filename}")
    print(f"{'='*60}")
    
    start_time = time.time()
    
    try:
        with rasterio.open(raster_path) as src:
            profile = src.profile.copy()
            profile.update(dtype=rasterio.float32, count=1, compress='lzw')
            
            pixel_size = abs(src.transform[0])
            window_size = int(round(2 * radius_m / pixel_size))
            if window_size % 2 == 0:
                window_size += 1
            
            print(f"  Window size: {window_size} pixels")
            print(f"  Loading raster...")
            raster_data = src.read(1)
            print(f"  Array shape: {raster_data.shape}")
            
            print(f"  Computing SHDI...")
            shdi_array = generic_filter(raster_data.astype(np.float32), calculate_shdi, size=window_size, mode='constant', cval=0)
            
            del raster_data
            gc.collect()
        
        print(f"  Writing output...")
        with rasterio.open(output_path, 'w', **profile) as dst:
            dst.write(shdi_array.astype(np.float32), 1)
        
        del shdi_array
        gc.collect()
        
        elapsed = time.time() - start_time
        print(f"  ✓ COMPLETED: {filename} ({elapsed:.1f}s)")
        return output_path
        
    except Exception as e:
        print(f"  ✗ ERROR: {filename} - {str(e)}")
        import traceback
        traceback.print_exc()
        return None

def main(input_folder, output_folder, radius_m=1000):
    print(f"\n{'#'*60}")
    print(f"SHDI PROCESSING STARTED")
    print(f"{'#'*60}")
    print(f"Input folder: {input_folder}")
    print(f"Output folder: {output_folder}")
    print(f"Radius: {radius_m}m")
    
    os.makedirs(output_folder, exist_ok=True)
    
    file_list = glob.glob(os.path.join(input_folder, '*.tif'))
    print(f"\nFound {len(file_list)} files to process")
    
    total_start = time.time()
    results = []
    
    for f in file_list:
        result = process_file(f, output_folder, radius_m)
        results.append(result)
    
    total_time = time.time() - total_start
    success_count = sum(1 for r in results if r is not None)
    
    print(f"\n{'#'*60}")
    print(f"PROCESSING COMPLETE: {success_count}/{len(file_list)} files")
    print(f"Total time: {total_time:.1f}s ({total_time/60:.1f}min)")
    print(f"{'#'*60}\n")

if __name__ == "__main__":
    main(INPUT_FOLDER, OUTPUT_FOLDER, RADIUS_M)



############################################################
SHDI PROCESSING STARTED
############################################################
Input folder: /home/gisuser/code/data/MB_v9_AtlanticForest_Landcover/orig_P_correct_nsf2_1.5_for_SHDI
Output folder: /home/gisuser/code/data/SHDI_landcover
Radius: 1000m

Found 3 files to process

Processing: MB_ls_2010_P_nsf2_1.5.tif
  Window size: 63 pixels
  Loading raster...
  Array shape: (22702, 28961)
  Computing SHDI...
  Writing output...
  ✓ COMPLETED: MB_ls_2010_P_nsf2_1.5.tif (14441.5s)

Processing: MB_ls_2015_P_nsf2_1.5.tif
  Window size: 63 pixels
  Loading raster...
  Array shape: (22702, 28961)
  Computing SHDI...
  Writing output...
  ✓ COMPLETED: MB_ls_2015_P_nsf2_1.5.tif (14666.2s)

Processing: MB_ls_2020_P_nsf2_1.5.tif
  Window size: 63 pixels
  Loading raster...
  Array shape: (22702, 28961)
  Computing SHDI...
  Writing output...
  ✓ COMPLETED: MB_ls_2020_P_nsf2_1.5.tif (14261.6s)

######################################

In [1]:
import os
import glob
import re
import time
import numpy as np
import rasterio
from scipy.ndimage import generic_filter
import gc

In [2]:
# Parameters
input_dir = r"/home/gisuser/code/data/MB_v9_AtlanticForest_Landcover/orig_P_correct_nsf2_1.5_for_SHDI"
output_dir = r"/home/gisuser/code/data/SHDI_landcover"
radius_meters = 1000

In [ ]:
if __name__ == "__main__":
    import time
    
    ## Run SHDI analysis stage
    print("Starting SHDI Analysis")
    shdi_start = time.time()
    process_shdi_analysis(input_dir, output_dir, radius_meters)
    shdi_duration = time.time() - shdi_start
    print(f"SHDI analysis completed in {shdi_duration:.2f} seconds")
    
    print("Processing complete!")

In [ ]:
# Calculate Shannon Diversity Index (SHDI) for landcover- maybe?

import os
import glob
import re
import time
import numpy as np
import dask.array as da
import rasterio
from skimage.util import view_as_windows
from dask.distributed import Client
import gc

##########
# Parameters
shdi_in = r"E:/NSF2_Naoya_Mikayla/Data/MB_v9_AtlanticForest_Landcover/nsf2_buff_1.5km"
shdi_out = r"E:/NSF2_Naoya_Mikayla/Data/SHDI_landcover"
shdi_radius = 1000  # meters

#########################################    

def get_year(filename):
    match = re.search(r"(\d{4})", filename)
    return match.group(1) if match else ""

def shdi_calculation(window):
    """Calculate SHDI for a single window"""
    # Remove nodata values (0)
    window = window[window != 0]
    if len(window) == 0:
        return np.nan
    
    # Get unique values and their counts
    vals, counts = np.unique(window, return_counts=True)
    # Calculate proportions
    ps = counts / counts.sum()
    # Calculate SHDI: -sum(p * ln(p))
    return -np.sum(ps * np.log(ps))

def process_windows_dask(windows_da):
    """Process windows using dask for parallel computation"""
    def apply_shdi(window_block):
        result = np.zeros(window_block.shape[:2], dtype=np.float32)
        for i in range(window_block.shape[0]):
            for j in range(window_block.shape[1]):
                window = window_block[i, j].flatten()
                result[i, j] = shdi_calculation(window)
        return result
    
    return da.map_blocks(apply_shdi, windows_da, 
                        drop_axis=[2, 3], 
                        dtype=np.float32)

def shdi_window_dask(input_raster, output_dir=shdi_out, radius=shdi_radius):
    """Calculate Shannon Diversity Index using dask and scikit-image"""
    try:
        basename = os.path.basename(input_raster)
        year = get_year(basename)
        output_path = os.path.join(output_dir, f"{year}_shdi.tif")
        
        if os.path.exists(output_path):
            print(f"Skipping {basename} - output already exists")
            return output_path
            
        print(f"Processing SHDI for {basename}")
        
        with rasterio.open(input_raster) as src:
            # Get raster properties
            profile = src.profile.copy()
            profile.update(dtype=rasterio.float32, nodata=np.nan)
            
            # Convert radius from meters to pixels
            pixel_size = abs(src.transform[0])
            pixel_radius = int(radius / pixel_size)
            window_size = pixel_radius * 2 + 1
            
            print(f"Raster size: {src.width} x {src.height}")
            print(f"Window size: {window_size} pixels ({radius}m radius)")
            
            # Read raster as dask array with chunking
            print("Loading raster with dask...")
            raster_data = da.from_array(src.read(1), chunks=(2048, 2048))
            
            # Create sliding windows using scikit-image
            print("Creating sliding windows...")
            # Pad array to handle edges
            padded_data = da.pad(raster_data, pixel_radius, mode='constant', constant_values=0)
            
            # Convert to numpy for view_as_windows (scikit-image doesn't support dask directly)
            padded_np = padded_data.compute()
            
            # Create sliding windows
            windows = view_as_windows(padded_np, (window_size, window_size), step=1)
            
            # Convert back to dask array with chunking
            windows_da = da.from_array(windows, chunks=(512, 512, window_size, window_size))
            
            print("Calculating SHDI with dask...")
            start_time = time.time()
            
            # Process windows in parallel
            shdi_result = process_windows_dask(windows_da)
            
            # Compute result
            shdi_computed = shdi_result.compute()
            
            calc_time = time.time() - start_time
            print(f"SHDI calculation completed in {calc_time:.2f} seconds")
            
            # Write output raster
            print("Writing output raster...")
            with rasterio.open(output_path, 'w', **profile) as dst:
                dst.write(shdi_computed, 1)
            
            # Clean up memory
            del raster_data, padded_data, windows, windows_da, shdi_result, shdi_computed
            gc.collect()
            
            print(f"SHDI successful: {output_path}")
            
        return output_path
        
    except Exception as e:
        print(f"SHDI error: {str(e)}")
        return None

def process_shdi_analysis(input_dir, output_dir, radius):
    """Process all raster files for SHDI calculation"""
    # Create output directory
    os.makedirs(output_dir, exist_ok=True)
    
    # Find all raster files
    pattern = os.path.join(input_dir, "*.tif")
    raster_files = glob.glob(pattern)
    
    if not raster_files:
        print(f"No raster files found in {input_dir}")
        return
    
    print(f"Found {len(raster_files)} raster files")
    
    # Process each raster
    results = []
    for input_path in raster_files:
        result = shdi_window_dask(input_path, output_dir, radius)
        results.append(result)
    
    success_count = sum(1 for r in results if r)
    print(f"Process complete: {success_count}/{len(raster_files)} succeeded")

# ==========
if __name__ == "__main__":
    # Initialize dask client
    client = Client(n_workers=4, threads_per_worker=2, memory_limit='4GB')
    print(f"Dask dashboard: {client.dashboard_link}")
    
    print("Starting Processing")
    
    ## Run SHDI stage
    print("Starting SHDI Analysis")
    shdi_start = time.time()
    process_shdi_analysis(shdi_in, shdi_out, shdi_radius)
    shdi_duration = time.time() - shdi_start
    print(f"SHDI analysis completed in {shdi_duration:.2f} seconds")
    
    print("Processing complete!")
    client.close()


In [ ]:
# perplexity version- maybe?
import os
import re
import time
import numpy as np
import rasterio
from skimage.util import view_as_windows
from glob import glob
from concurrent.futures import ProcessPoolExecutor, as_completed
import multiprocessing

##########
# Parameters
shdi_in = r"E:/NSF2_Naoya_Mikayla/Data/MB_v9_AtlanticForest_Landcover/nsf2_buff_1.5km"
shdi_out = r"E:/NSF2_Naoya_Mikayla/Data/SHDI_landcover"
shdi_radius = 1000  # meters
cores = multiprocessing.cpu_count()

os.makedirs(shdi_out, exist_ok=True)

#########################################

def get_year(filename):
    """Extract year from filename"""
    match = re.search(r"(\d{4})", filename)
    return match.group(1) if match else ""


def calculate_shdi_window(window):
    """Calculate Shannon Diversity Index for a single window"""
    # Filter out nodata (0)
    window_flat = window[window != 0]
    
    if len(window_flat) == 0:
        return 0.0
    
    # Count unique values and calculate proportions
    vals, counts = np.unique(window_flat, return_counts=True)
    ps = counts / counts.sum()
    
    # SHDI formula: -sum(p * log(p))
    ps_log = np.where(ps > 0, np.log(ps), 0)
    shdi = -np.sum(ps * ps_log)
    
    return shdi


def shdi_window_skimage(input_raster, output_dir=shdi_out, radius=shdi_radius):
    """Calculate Shannon Diversity Index using scikit-image moving window"""
    try:
        basename = os.path.basename(input_raster)
        year = get_year(basename)
        output_path = os.path.join(output_dir, f"{year}_shdi.tif")
        
        if os.path.exists(output_path):
            print(f"Skipping existing: {output_path}")
            return output_path
        
        print(f"Processing SHDI for {basename}")
        
        # 1. Read raster with rasterio
        with rasterio.open(input_raster) as src:
            arr = src.read(1)
            profile = src.profile.copy()
            cell_size = src.res[0]
            
        # 2. Convert radius from meters to pixels
        pixel_radius = int(radius / cell_size)
        window_size = 2 * pixel_radius + 1
        
        print(f"  Window size: {window_size}x{window_size} pixels ({radius}m radius)")
        
        # 3. Pad array to handle edges (reflect mode preserves patterns at boundaries)
        pad_width = pixel_radius
        arr_padded = np.pad(arr, pad_width, mode='reflect')
        
        # 4. Create sliding windows view (no data copying, just views)
        print(f"  Creating sliding windows...")
        windows = view_as_windows(arr_padded, (window_size, window_size))
        
        # 5. Calculate SHDI for each position using vectorized iteration
        print(f"  Calculating SHDI for {windows.shape[0]}x{windows.shape[1]} windows...")
        rows, cols = windows.shape[0], windows.shape[1]
        output_arr = np.zeros((rows, cols), dtype=np.float32)
        
        # Vectorized calculation over all windows
        for i in range(rows):
            for j in range(cols):
                output_arr[i, j] = calculate_shdi_window(windows[i, j])
            
            # Progress indicator
            if (i + 1) % 100 == 0:
                print(f"    Progress: {i+1}/{rows} rows completed")
        
        # 6. Write output raster
        print(f"  Writing output: {output_path}")
        profile.update(dtype=rasterio.float32, count=1, nodata=0)
        
        with rasterio.open(output_path, 'w', **profile) as dst:
            dst.write(output_arr, 1)
        
        print(f"  SHDI successful: {output_path}")
        return output_path
        
    except Exception as e:
        print(f"  SHDI error for {basename}: {str(e)}")
        import traceback
        traceback.print_exc()
        return None


def process_all_rasters_parallel(input_dir, output_dir, radius, max_workers=None):
    """Process all rasters in parallel using ProcessPoolExecutor"""
    rasters = sorted(glob(os.path.join(input_dir, "*.tif")))
    
    if not rasters:
        print(f"No rasters found in: {input_dir}")
        return []
    
    if max_workers is None:
        max_workers = cores
    
    print(f"Found {len(rasters)} rasters to process")
    print(f"Using {max_workers} parallel workers\n")
    
    results = []
    
    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        # Submit all tasks
        future_to_raster = {
            executor.submit(shdi_window_skimage, raster, output_dir, radius): raster 
            for raster in rasters
        }
        
        # Process completed tasks
        for future in as_completed(future_to_raster):
            raster = future_to_raster[future]
            try:
                result = future.result()
                results.append(result)
            except Exception as e:
                print(f"Error processing {os.path.basename(raster)}: {e}")
                results.append(None)
    
    success_count = sum(1 for r in results if r)
    print(f"\nProcess complete: {success_count}/{len(rasters)} succeeded")
    return results


#########################################

if __name__ == "__main__":
    print("Starting SHDI Processing with scikit-image moving window")
    print(f"Input: {shdi_in}")
    print(f"Output: {shdi_out}")
    print(f"Radius: {shdi_radius}m")
    print(f"CPU cores: {cores}\n")
    
    start_time = time.time()
    
    # Process rasters in parallel (each raster gets its own process)
    results = process_all_rasters_parallel(shdi_in, shdi_out, shdi_radius)
    
    duration = time.time() - start_time
    print(f"\nTotal processing time: {duration:.2f} seconds ({duration/60:.2f} minutes)")
    print("Processing complete!")
